# Recreate KaroSpace `Compare > Simple design`

This notebook recreates the feature-level tables behind `Statistics > Compare > Simple design` for both methods shown by the KaroSpace method selector:

- **Wilcoxon**: cell-level category-vs-category rank-sum contrast.
- **Pseudobulk DESeq2**: replicate × category pseudobulk aggregation, one shared `~ replicate + annotation` model, then the selected category-vs-category contrast.

The notebook does not call KaroSpace viewer or HTML functions. It expands the calculation path so the intermediate matrices, filters, fitted samples, p-values, log2FC values, and display thresholds can be inspected.

## What The Simple Design Panel Displays

For the selected `Annotation A` and `Annotation B`, the viewer reads one contrast result from either `wilcoxon_de_by_modality` or `pseudobulk_de_by_modality`.

The **raw table** filters that result to rows with `padj < cutoff` and `abs(log2FC) >= cutoff`. Positive log2FC means the feature is enriched in Annotation A; negative log2FC means enriched in Annotation B.

The **Features** view uses the same result arrays for MA and volcano plots: `features`, `base_mean`, `log2foldchanges`, `pvals`, `pvals_adj`, `% A`, and `% B`.

In [ ]:
from pathlib import Path
import inspect
import math
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse, stats

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "karospace").is_dir():
        repo_root = candidate
        break


## Inputs

In [ ]:
h5ad_path = repo_root / "tests" / "comet_xenium_multimodal.h5ad"

# Modality. Use "rna" for adata.X. For the bundled test data, "protein" maps to adata.obsm["protein"].
modality = "rna"
obsm_modality_var_keys = {"protein": "protein_var"}

# Simple design contrast.
annotation_col = "leiden_rna"
source_category = "0"      # Annotation A
reference_category = "1"   # Annotation B
replicate_col = "sample_id"

# Matrix/statistics settings.
wilcoxon_layer = None
statistics_counts_layer = "counts"
statistics_min_cell_counts = 0
statistics_min_feature_counts = 0

# Wilcoxon settings.
wilcoxon_min_cells_per_group = 20
wilcoxon_min_pct_expressed = 0.0
wilcoxon_p_adjust_method = "fdr_bh"
wilcoxon_padj_cutoff = 0.05
wilcoxon_log2fc_cutoff = 1.0
wilcoxon_top_n_per_category = 300
wilcoxon_feature_chunk_size = 128

# Pseudobulk DESeq2 settings.
run_deseq2 = True
pseudobulk_min_cells_per_pseudobulk = 20
pseudobulk_min_replicates = 2
pseudobulk_min_pct_expressed = 0.0
pseudobulk_p_adjust_method = "fdr_bh"
pseudobulk_padj_cutoff = 0.05
pseudobulk_log2fc_cutoff = 0.5
pseudobulk_fit_type = "parametric"
pseudobulk_n_cpus = 1


## Shared Helpers

In [ ]:
TARGET_SUM = 10000.0
LOG2FC_EPSILON = 1e-9


def compact_float(value, significant_digits=6):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return None
    if not np.isfinite(value):
        return None
    return float(f"{value:.{max(1, int(significant_digits))}g}")


def normalize_pct_threshold(value):
    threshold = float(value or 0.0)
    if threshold > 1.0:
        threshold = threshold / 100.0
    return min(max(threshold, 0.0), 1.0)


def adjust_pvalues(pvalues, method="fdr_bh"):
    p = np.asarray(pvalues, dtype=float)
    adjusted = np.full(p.shape, np.nan, dtype=float)
    finite = np.isfinite(p)
    if not finite.any():
        return adjusted
    idx = np.flatnonzero(finite)
    vals = np.clip(p[idx], 0, 1)
    method_norm = str(method or "fdr_bh").strip().lower().replace("-", "_")
    if method_norm in {"none", "raw", "pvalue", "pvalues"}:
        adjusted[idx] = vals
    elif method_norm in {"bonferroni", "bonf"}:
        adjusted[idx] = np.clip(vals * len(vals), 0, 1)
    else:
        order = np.argsort(vals)
        ranked = vals[order]
        n = len(ranked)
        if method_norm in {"holm", "holm_bonferroni"}:
            adj = (n - np.arange(n, dtype=float)) * ranked
            adj = np.maximum.accumulate(adj)
        else:
            adj = ranked * n / (np.arange(n, dtype=float) + 1.0)
            adj = np.minimum.accumulate(adj[::-1])[::-1]
        adjusted[idx[order]] = np.clip(adj, 0, 1)
    adjusted[(adjusted == 0) & np.isfinite(adjusted)] = np.nextafter(0.0, 1.0)
    return adjusted


def matrix_axis_sum(matrix, axis):
    values = np.asarray(matrix.sum(axis=axis)).ravel() if sparse.issparse(matrix) else np.asarray(matrix, dtype=float).sum(axis=axis)
    values = np.asarray(values, dtype=float).ravel()
    values[~np.isfinite(values)] = 0.0
    return values


def copy_matrix(matrix):
    return matrix.copy() if sparse.issparse(matrix) else np.array(matrix, dtype=float, copy=True)


def library_size_normalize(matrix, target_sum=TARGET_SUM):
    matrix = copy_matrix(matrix)
    if sparse.issparse(matrix):
        matrix = matrix.astype(np.float64, copy=False).tocsr()
        matrix.data[~np.isfinite(matrix.data)] = 0.0
        row_sums = np.asarray(matrix.sum(axis=1)).ravel()
        scale = np.divide(float(target_sum), row_sums, out=np.zeros_like(row_sums, dtype=float), where=row_sums > 0)
        normalized = sparse.diags(scale).dot(matrix).tocsr()
        normalized.data[~np.isfinite(normalized.data)] = 0.0
        normalized.eliminate_zeros()
        return normalized
    matrix[~np.isfinite(matrix)] = 0.0
    row_sums = matrix.sum(axis=1)
    scale = np.divide(float(target_sum), row_sums, out=np.zeros_like(row_sums, dtype=float), where=row_sums > 0)
    matrix *= scale[:, None]
    matrix[~np.isfinite(matrix)] = 0.0
    return matrix


def log_normalize(matrix, target_sum=TARGET_SUM):
    normalized = library_size_normalize(matrix, target_sum=target_sum)
    if sparse.issparse(normalized):
        normalized = normalized.tocsr(copy=True)
        normalized.data = np.log1p(normalized.data)
        normalized.data[~np.isfinite(normalized.data)] = 0.0
        normalized.eliminate_zeros()
        return normalized
    normalized = np.log1p(normalized)
    normalized[~np.isfinite(normalized)] = 0.0
    return normalized


def matrix_to_dense(matrix):
    return matrix.toarray() if sparse.issparse(matrix) else np.asarray(matrix, dtype=float)


def column_means(matrix, row_mask):
    if not bool(np.any(row_mask)):
        return np.zeros(int(matrix.shape[1]), dtype=float)
    subset = matrix[row_mask]
    means = np.asarray(subset.mean(axis=0), dtype=float).ravel() if sparse.issparse(subset) else np.asarray(subset, dtype=float).mean(axis=0)
    means = np.asarray(means, dtype=float)
    means[~np.isfinite(means)] = 0.0
    return means


def positive_fraction(matrix, row_mask, feature_indices):
    if not bool(np.any(row_mask)):
        return [None for _ in feature_indices]
    subset = matrix[row_mask]
    if sparse.issparse(subset):
        subset = subset[:, list(feature_indices)]
        counts = np.asarray((subset > 0).sum(axis=0)).ravel()
    else:
        subset = np.asarray(subset, dtype=float)[:, list(feature_indices)]
        counts = np.count_nonzero(subset > 0, axis=0)
    return [float(value) / int(row_mask.sum()) for value in counts]


def display_table_from_result(result, padj_cutoff, log2fc_cutoff):
    rows = []
    features = result.get("features") or []
    for i, feature in enumerate(features):
        row = {
            "feature": feature,
            "log2fc": (result.get("log2foldchanges") or [np.nan] * len(features))[i],
            "pval": (result.get("pvals") or [np.nan] * len(features))[i],
            "padj": (result.get("pvals_adj") or [np.nan] * len(features))[i],
            "score": (result.get("scores") or [np.nan] * len(features))[i],
            "pct_A": (result.get("pct_source") or [np.nan] * len(features))[i],
            "pct_B": (result.get("pct_reference") or [np.nan] * len(features))[i],
            "base_mean": (result.get("base_mean") or [np.nan] * len(features))[i],
        }
        try:
            keep = float(row["padj"]) < float(padj_cutoff) and abs(float(row["log2fc"])) >= float(log2fc_cutoff)
        except (TypeError, ValueError):
            keep = False
        if keep:
            rows.append(row)
    return pd.DataFrame(rows)


## Load Data And Resolve Matrices

In [ ]:
def make_modality_adata(adata, modality_name, obsm_var_keys=None):
    modality_name = str(modality_name)
    obsm_var_keys = obsm_var_keys or {}
    if modality_name == "rna":
        return adata
    if modality_name not in adata.obsm:
        raise KeyError(f"{modality_name!r} is not an obsm matrix. Available: {list(adata.obsm.keys())}")
    matrix = adata.obsm[modality_name]
    var_key = obsm_var_keys.get(modality_name, f"{modality_name}_var")
    var_table = adata.uns.get(var_key)
    if isinstance(var_table, pd.DataFrame) and not var_table.empty:
        var = var_table.copy()
        var.index = var.iloc[:, 0].astype(str).to_numpy()
    else:
        var = pd.DataFrame(index=[str(i) for i in range(int(matrix.shape[1]))])
    return ad.AnnData(X=matrix, obs=adata.obs.copy(), var=var)


def resolve_wilcoxon_matrix(adata, expression_layer=None):
    layers = getattr(adata, "layers", None) or {}
    if expression_layer and expression_layer in layers:
        if str(expression_layer) == "normalized":
            return layers[expression_layer], str(expression_layer)
        return log_normalize(layers[expression_layer]), f"{expression_layer}_log1p_normalized"
    if "normalized" in layers:
        return layers["normalized"], "normalized"
    if "counts" in layers:
        return log_normalize(layers["counts"]), "counts_log1p_normalized"
    return log_normalize(adata.X), "X_log1p_normalized"


def resolve_counts_matrix(adata, counts_layer="counts"):
    layers = getattr(adata, "layers", None) or {}
    if counts_layer and counts_layer in layers:
        return layers[counts_layer], str(counts_layer)
    if counts_layer:
        print(f"statistics_counts_layer={counts_layer!r} was not found; using adata.X.")
    return adata.X, "X"


adata = ad.read_h5ad(h5ad_path)
analysis_adata = make_modality_adata(adata, modality, obsm_modality_var_keys)
feature_names = [str(feature) for feature in analysis_adata.var_names]

wilcoxon_matrix, wilcoxon_matrix_source = resolve_wilcoxon_matrix(analysis_adata, wilcoxon_layer)
count_matrix, counts_layer_used = resolve_counts_matrix(analysis_adata, statistics_counts_layer)

if annotation_col not in analysis_adata.obs:
    raise KeyError(f"{annotation_col!r} is not an obs column")
if replicate_col not in analysis_adata.obs:
    raise KeyError(f"{replicate_col!r} is not an obs column")

annotation = analysis_adata.obs[annotation_col]
if pd.api.types.is_numeric_dtype(annotation):
    raise TypeError("Simple design requires a categorical annotation")
if not isinstance(annotation.dtype, pd.CategoricalDtype):
    annotation = annotation.astype("category")
labels_all = annotation.astype(str).to_numpy()
categories = [str(category) for category in annotation.cat.categories]

if source_category not in categories or reference_category not in categories:
    raise ValueError(f"source/reference must be in {categories}")

print(f"Loaded {analysis_adata.n_obs:,} cells x {analysis_adata.n_vars:,} features")
print(f"Wilcoxon matrix: {wilcoxon_matrix_source}")
print(f"Count matrix: {counts_layer_used}")
print(f"Contrast: {source_category} vs {reference_category}")


## Wilcoxon Simple Design

KaroSpace's Simple design Wilcoxon result is a category-vs-category contrast. Unlike the marker list category-vs-rest path, the result is two-sided for log2FC: the display threshold is `abs(log2FC) >= cutoff`.

In [ ]:
def wilcoxon_pairwise_rank_table(matrix, labels, source, reference, feature_names, chunk_size=128):
    source_mask = labels == str(source)
    reference_mask = labels == str(reference)
    pair_mask = source_mask | reference_mask
    pair_labels = labels[pair_mask]
    pair_matrix = matrix[pair_mask]
    pair_source_mask = pair_labels == str(source)
    pair_reference_mask = pair_labels == str(reference)
    rows = []
    for start in range(0, int(pair_matrix.shape[1]), int(chunk_size)):
        end = min(start + int(chunk_size), int(pair_matrix.shape[1]))
        block = matrix_to_dense(pair_matrix[:, start:end])
        source_block = block[pair_source_mask]
        reference_block = block[pair_reference_mask]
        if source_block.shape[0] == 0 or reference_block.shape[0] == 0:
            scores = np.zeros(end - start, dtype=float)
            pvalues = np.ones(end - start, dtype=float)
        else:
            try:
                test = stats.ranksums(source_block, reference_block, axis=0, nan_policy="propagate")
                scores = np.asarray(test.statistic, dtype=float)
                pvalues = np.asarray(test.pvalue, dtype=float)
            except TypeError:
                chunk_scores = []
                chunk_pvalues = []
                for offset in range(end - start):
                    test = stats.ranksums(source_block[:, offset], reference_block[:, offset])
                    chunk_scores.append(test.statistic)
                    chunk_pvalues.append(test.pvalue)
                scores = np.asarray(chunk_scores, dtype=float)
                pvalues = np.asarray(chunk_pvalues, dtype=float)
        scores[~np.isfinite(scores)] = 0.0
        pvalues[~np.isfinite(pvalues)] = 1.0
        pvalues = np.clip(pvalues, 0, 1)
        for offset, feature in enumerate(feature_names[start:end]):
            rows.append({"feature": str(feature), "score": float(scores[offset]), "pvalue": float(pvalues[offset])})
    return pd.DataFrame(rows)


def format_wilcoxon_pairwise_result(rank_table, matrix, labels, source, reference, feature_names):
    source_mask = labels == str(source)
    reference_mask = labels == str(reference)
    if int(source_mask.sum()) < int(wilcoxon_min_cells_per_group) or int(reference_mask.sum()) < int(wilcoxon_min_cells_per_group):
        raise ValueError("not enough cells in source or reference for Wilcoxon")
    feature_to_idx = {str(feature): i for i, feature in enumerate(feature_names)}
    work = rank_table.copy()
    work["_feature"] = work["feature"].astype(str)
    work["_feature_idx"] = [feature_to_idx.get(feature) for feature in work["_feature"]]
    work = work[work["_feature_idx"].notna()].copy()
    indices = [int(i) for i in work["_feature_idx"]]
    source_means = column_means(matrix, source_mask)
    reference_means = column_means(matrix, reference_mask)
    base_means = column_means(matrix, source_mask | reference_mask)
    pvals = pd.to_numeric(work["pvalue"], errors="coerce").to_numpy(dtype=float)
    pvals[~np.isfinite(pvals)] = 1.0
    pvals = np.clip(pvals, 0, 1)
    padj = adjust_pvalues(pvals, wilcoxon_p_adjust_method)
    pct_source = positive_fraction(matrix, source_mask, indices)
    pct_reference = positive_fraction(matrix, reference_mask, indices)
    work["_pvalue"] = pvals
    work["_padj"] = padj
    work["_log2fc"] = np.log2(source_means[indices] + LOG2FC_EPSILON) - np.log2(reference_means[indices] + LOG2FC_EPSILON)
    work["_score"] = pd.to_numeric(work["score"], errors="coerce").fillna(0).to_numpy(dtype=float)
    work["_pct_source"] = [float(v) if v is not None and np.isfinite(v) else 0.0 for v in pct_source]
    work["_pct_reference"] = [float(v) if v is not None and np.isfinite(v) else 0.0 for v in pct_reference]
    work["_base_mean"] = base_means[indices]
    min_pct = normalize_pct_threshold(wilcoxon_min_pct_expressed)
    if min_pct > 0:
        work = work[(work["_pct_source"] >= min_pct) | (work["_pct_reference"] >= min_pct)].copy()
    work = work[np.isfinite(work["_log2fc"].to_numpy(dtype=float))].copy()
    work["_abs_lfc"] = np.abs(work["_log2fc"].to_numpy(dtype=float))
    work = work.sort_values(["_padj", "_pvalue", "_abs_lfc", "_feature"], ascending=[True, True, False, True])
    work = work.head(max(1, int(wilcoxon_top_n_per_category)))
    return {
        "available": True,
        "method": "cell-wilcoxon-pairwise-expanded-notebook",
        "p_adjust_method": wilcoxon_p_adjust_method,
        "min_pct_expressed": min_pct,
        "padj_cutoff": float(wilcoxon_padj_cutoff),
        "log2fc_cutoff": float(wilcoxon_log2fc_cutoff),
        "n_source": int(source_mask.sum()),
        "n_reference": int(reference_mask.sum()),
        "n_replicates": 0,
        "counts_layer": wilcoxon_matrix_source,
        "features": work["_feature"].astype(str).tolist(),
        "log2foldchanges": [compact_float(v, 6) for v in work["_log2fc"]],
        "pvals": [compact_float(v, 6) for v in work["_pvalue"]],
        "pvals_adj": [compact_float(v, 6) for v in work["_padj"]],
        "scores": [compact_float(v, 6) for v in work["_score"]],
        "pct_source": [compact_float(v, 5) for v in work["_pct_source"]],
        "pct_reference": [compact_float(v, 5) for v in work["_pct_reference"]],
        "base_mean": [compact_float(v, 6) for v in work["_base_mean"]],
    }


cell_count_totals = matrix_axis_sum(count_matrix, axis=1)
cell_filter = np.ones(analysis_adata.n_obs, dtype=bool)
if int(statistics_min_cell_counts) > 0:
    cell_filter &= np.isfinite(cell_count_totals) & (cell_count_totals >= int(statistics_min_cell_counts))
feature_filter = np.ones(analysis_adata.n_vars, dtype=bool)
if int(statistics_min_feature_counts) > 0:
    feature_totals = matrix_axis_sum(count_matrix[cell_filter], axis=0)
    feature_filter &= np.isfinite(feature_totals) & (feature_totals >= int(statistics_min_feature_counts))

wilcoxon_labels = labels_all[cell_filter]
wilcoxon_filtered_matrix = wilcoxon_matrix[cell_filter][:, feature_filter]
wilcoxon_features = [feature for feature, keep in zip(feature_names, feature_filter) if bool(keep)]

rank_table = wilcoxon_pairwise_rank_table(
    wilcoxon_filtered_matrix,
    wilcoxon_labels,
    source_category,
    reference_category,
    wilcoxon_features,
    chunk_size=wilcoxon_feature_chunk_size,
)
wilcoxon_result = format_wilcoxon_pairwise_result(rank_table, wilcoxon_filtered_matrix, wilcoxon_labels, source_category, reference_category, wilcoxon_features)
wilcoxon_simple_design_table = display_table_from_result(wilcoxon_result, wilcoxon_padj_cutoff, wilcoxon_log2fc_cutoff)

print(f"Wilcoxon returned {len(wilcoxon_result['features']):,} rows before Simple design display thresholds")
print(f"Wilcoxon display table has {len(wilcoxon_simple_design_table):,} rows")
wilcoxon_simple_design_table.head(20)


## Pseudobulk DESeq2 Simple Design

This reproduces the standard cell-annotation model: one pseudobulk sample per `replicate_col × annotation_col`, model `~ _pb_replicate + _pb_group`, then `source_category - reference_category`.

In [ ]:
def to_dense_counts(matrix):
    dense = matrix.toarray() if sparse.issparse(matrix) else np.asarray(matrix)
    dense = np.asarray(dense, dtype=np.float64)
    dense[~np.isfinite(dense)] = 0
    dense[dense < 0] = 0
    return np.rint(dense).astype(np.int64, copy=False)


def aggregate_replicate_by_category(count_matrix, obs, replicate_col, annotation_values, valid_mask):
    valid_indices = np.flatnonzero(valid_mask)
    reps = obs[replicate_col].astype(str).iloc[valid_indices].to_numpy()
    groups = annotation_values[valid_indices]
    sample_keys = []
    sample_index = {}
    row_ids = np.empty(valid_indices.size, dtype=np.int64)
    for i, key in enumerate(zip(reps, groups)):
        key = (str(key[0]), str(key[1]))
        if key not in sample_index:
            sample_index[key] = len(sample_keys)
            sample_keys.append(key)
        row_ids[i] = sample_index[key]
    incidence = sparse.csr_matrix((np.ones(valid_indices.size), (row_ids, valid_indices)), shape=(len(sample_keys), count_matrix.shape[0]))
    counts = to_dense_counts(incidence @ count_matrix)
    meta = pd.DataFrame({
        "_pb_replicate": [key[0] for key in sample_keys],
        "_pb_group": [key[1] for key in sample_keys],
        "n_cells": np.bincount(row_ids, minlength=len(sample_keys)).astype(int),
    }, index=[f"pb_{i}" for i in range(len(sample_keys))])
    return counts, meta


def filter_pseudobulk_features(counts, feature_names, min_feature_counts):
    if int(min_feature_counts) <= 0:
        return counts, list(feature_names)
    totals = np.asarray(counts, dtype=float).sum(axis=0)
    keep = np.isfinite(totals) & (totals >= int(min_feature_counts))
    return counts[:, keep], [feature for feature, ok in zip(feature_names, keep) if bool(ok)]


def shared_design_rank(metadata):
    reps = pd.get_dummies(metadata["_pb_replicate"].astype(str), drop_first=True, dtype=float)
    groups = pd.get_dummies(metadata["_pb_group"].astype(str), drop_first=True, dtype=float)
    design = pd.concat([pd.Series(1.0, index=metadata.index, name="Intercept"), reps, groups], axis=1)
    matrix = design.to_numpy(dtype=float)
    return int(np.linalg.matrix_rank(matrix)), int(matrix.shape[1]), design.columns.tolist()


def fit_deseq2_shared(counts, metadata, feature_names, retained_categories):
    from pydeseq2.dds import DeseqDataSet
    counts_df = pd.DataFrame(counts, index=metadata.index, columns=feature_names)
    design_meta = metadata[["_pb_replicate", "_pb_group"]].copy()
    design_meta["_pb_replicate"] = pd.Categorical(design_meta["_pb_replicate"].astype(str))
    design_meta["_pb_group"] = pd.Categorical(design_meta["_pb_group"].astype(str), categories=[str(c) for c in retained_categories])
    try:
        dds = DeseqDataSet(counts=counts_df, metadata=design_meta, design="~ _pb_replicate + _pb_group", fit_type=pseudobulk_fit_type, n_cpus=max(1, int(pseudobulk_n_cpus)), quiet=True)
    except TypeError:
        dds = DeseqDataSet(counts=counts_df, clinical=design_meta, design_factors=["_pb_replicate", "_pb_group"], fit_type=pseudobulk_fit_type, refit_cooks=True, n_cpus=max(1, int(pseudobulk_n_cpus)))
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        dds.deseq2()
    return dds


def run_deseq2_pairwise(dds, source, reference):
    from pydeseq2.ds import DeseqStats
    contrast = np.asarray(dds.contrast(column="_pb_group", baseline=str(reference), group_to_compare=str(source)), dtype=float)
    try:
        kwargs = {"dds": dds, "contrast": contrast, "quiet": True, "n_cpus": 1, "independent_filter": False}
        if "independent_filter" not in inspect.signature(DeseqStats.__init__).parameters:
            kwargs.pop("independent_filter")
        stat_res = DeseqStats(**kwargs)
    except TypeError:
        stat_res = DeseqStats(dds, contrast=contrast, n_cpus=1)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        stat_res.summary()
    return stat_res.results_df.copy()


def expression_prefilter_features(count_matrix, source_mask, reference_mask, all_features, fitted_features, min_pct):
    threshold = normalize_pct_threshold(min_pct)
    fitted_features = [str(f) for f in fitted_features]
    if threshold <= 0:
        return fitted_features
    feature_to_idx = {str(f): i for i, f in enumerate(all_features)}
    fitted_indices = [feature_to_idx[f] for f in fitted_features if f in feature_to_idx]
    pct_source = positive_fraction(count_matrix, source_mask, fitted_indices)
    pct_reference = positive_fraction(count_matrix, reference_mask, fitted_indices)
    keep = []
    for feature, ps, pr in zip(fitted_features, pct_source, pct_reference):
        if max(float(ps or 0), float(pr or 0)) >= threshold:
            keep.append(feature)
    return keep


def format_deseq2_result(raw_df, source_mask, reference_mask, raw_count_matrix, all_feature_names, fitted_feature_names):
    work = raw_df.copy()
    if "log2FoldChange" not in work:
        work["log2FoldChange"] = np.nan
    if "pvalue" not in work:
        work["pvalue"] = np.nan
    if "stat" not in work:
        work["stat"] = np.nan
    if "baseMean" not in work:
        work["baseMean"] = np.nan
    work = work.reindex([f for f in fitted_feature_names if f in work.index]).copy()
    work["padj"] = adjust_pvalues(work["pvalue"].to_numpy(dtype=float), pseudobulk_p_adjust_method)
    work["_feature"] = work.index.astype(str)
    work = work[np.isfinite(work["log2FoldChange"].to_numpy(dtype=float))].copy()
    feature_to_idx = {str(f): i for i, f in enumerate(all_feature_names)}
    work["_feature_idx"] = [feature_to_idx.get(f) for f in work["_feature"]]
    work = work[work["_feature_idx"].notna()].copy()
    indices = [int(i) for i in work["_feature_idx"]]
    pct_source = positive_fraction(raw_count_matrix, source_mask, indices)
    pct_reference = positive_fraction(raw_count_matrix, reference_mask, indices)
    work["_pct_source"] = pct_source
    work["_pct_reference"] = pct_reference
    work["_padj_sort"] = work["padj"].fillna(np.inf)
    work["_pvalue_sort"] = work["pvalue"].fillna(np.inf)
    work["_abs_lfc"] = np.abs(work["log2FoldChange"].to_numpy(dtype=float))
    work = work.sort_values(["_padj_sort", "_pvalue_sort", "_abs_lfc", "_feature"], ascending=[True, True, False, True])
    return {
        "available": True,
        "method": "pseudobulk-deseq2-expanded-notebook",
        "p_adjust_method": pseudobulk_p_adjust_method,
        "min_pct_expressed": normalize_pct_threshold(pseudobulk_min_pct_expressed),
        "padj_cutoff": float(pseudobulk_padj_cutoff),
        "log2fc_cutoff": float(pseudobulk_log2fc_cutoff),
        "n_source": int(source_mask.sum()),
        "n_reference": int(reference_mask.sum()),
        "n_replicates": int(len(set(analysis_adata.obs.loc[source_mask, replicate_col].astype(str)) & set(analysis_adata.obs.loc[reference_mask, replicate_col].astype(str)))),
        "counts_layer": counts_layer_used,
        "features": work["_feature"].astype(str).tolist(),
        "log2foldchanges": [compact_float(v, 6) for v in work["log2FoldChange"]],
        "pvals": [compact_float(v, 6) for v in work["pvalue"]],
        "pvals_adj": [compact_float(v, 6) for v in work["padj"]],
        "scores": [compact_float(v, 6) for v in work["stat"]],
        "pct_source": [compact_float(v, 5) for v in work["_pct_source"]],
        "pct_reference": [compact_float(v, 5) for v in work["_pct_reference"]],
        "base_mean": [compact_float(v, 6) for v in work["baseMean"]],
    }


In [ ]:
deseq2_result = None
deseq2_simple_design_table = pd.DataFrame()

if run_deseq2:
    try:
        import pydeseq2  # noqa: F401
    except ImportError as exc:
        raise ImportError("Install pydeseq2 to run the DESeq2 section") from exc

    replicate_values = analysis_adata.obs[replicate_col].astype(str)
    valid = replicate_values.notna().to_numpy() & pd.notna(labels_all)
    valid &= np.asarray(annotation.cat.codes.to_numpy() >= 0, dtype=bool)
    if int(statistics_min_cell_counts) > 0:
        cell_totals = matrix_axis_sum(count_matrix, axis=1)
        valid &= np.isfinite(cell_totals) & (cell_totals >= int(statistics_min_cell_counts))

    pb_counts, pb_meta = aggregate_replicate_by_category(count_matrix, analysis_adata.obs, replicate_col, labels_all, valid)
    pb_counts, pb_features = filter_pseudobulk_features(pb_counts, feature_names, statistics_min_feature_counts)
    pb_meta.attrs["feature_names"] = pb_features

    sufficient = pb_meta[pb_meta["n_cells"] >= int(pseudobulk_min_cells_per_pseudobulk)]
    retained_categories = [
        category for category in categories
        if sufficient.loc[sufficient["_pb_group"] == category, "_pb_replicate"].nunique() >= max(2, int(pseudobulk_min_replicates))
    ]
    if source_category not in retained_categories or reference_category not in retained_categories:
        raise ValueError(f"source/reference not retained for DESeq2. Retained categories: {retained_categories}")

    model_mask = pb_meta["_pb_group"].isin(retained_categories) & (pb_meta["n_cells"] >= int(pseudobulk_min_cells_per_pseudobulk))
    model_counts = pb_counts[np.flatnonzero(model_mask.to_numpy())]
    model_meta = pb_meta.loc[model_mask].copy()
    rank, n_columns, design_columns = shared_design_rank(model_meta)
    residual_df = int(len(model_meta) - rank)
    if rank < n_columns or residual_df <= 0:
        raise ValueError(f"DESeq2 design is rank-deficient or has no residual df: rank {rank}/{n_columns}, residual df {residual_df}")

    print(f"Pseudobulk samples before model filter: {len(pb_meta):,}")
    print(f"Model samples: {len(model_meta):,}; features: {model_counts.shape[1]:,}; design rank {rank}/{n_columns}; residual df {residual_df}")
    display(model_meta.groupby("_pb_group").agg(n_pseudobulk=("n_cells", "size"), n_replicates=("_pb_replicate", "nunique"), cells=("n_cells", "sum")))

    dds = fit_deseq2_shared(model_counts, model_meta, pb_features, retained_categories)
    source_mask = valid & (labels_all == str(source_category))
    reference_mask = valid & (labels_all == str(reference_category))
    fitted_features = [str(feature) for feature in pb_features]
    test_features = expression_prefilter_features(count_matrix, source_mask, reference_mask, feature_names, fitted_features, pseudobulk_min_pct_expressed)
    raw_deseq2 = run_deseq2_pairwise(dds, source_category, reference_category).reindex(test_features).dropna(how="all")
    deseq2_result = format_deseq2_result(raw_deseq2, source_mask, reference_mask, count_matrix, feature_names, test_features)
    deseq2_simple_design_table = display_table_from_result(deseq2_result, pseudobulk_padj_cutoff, pseudobulk_log2fc_cutoff)
    print(f"DESeq2 returned {len(deseq2_result['features']):,} rows before Simple design display thresholds")
    print(f"DESeq2 display table has {len(deseq2_simple_design_table):,} rows")

deseq2_simple_design_table.head(20)


## Compare The Two Simple Design Tables

In [ ]:
wilcoxon_labeled = wilcoxon_simple_design_table.copy()
wilcoxon_labeled.insert(0, "method", "wilcoxon")
deseq2_labeled = deseq2_simple_design_table.copy()
if not deseq2_labeled.empty:
    deseq2_labeled.insert(0, "method", "deseq2")

combined_simple_design = pd.concat([wilcoxon_labeled, deseq2_labeled], ignore_index=True, sort=False)
display(combined_simple_design.head(40))

overlap = sorted(set(wilcoxon_simple_design_table.get("feature", [])) & set(deseq2_simple_design_table.get("feature", [])))
print(f"Significant feature overlap: {len(overlap):,}")
pd.DataFrame({"overlap_feature": overlap[:50]})


## Recreate The Feature Plot Inputs

In [ ]:
def result_dataframe(result):
    features = result.get("features") or []
    return pd.DataFrame({
        "feature": features,
        "base_mean": result.get("base_mean") or [np.nan] * len(features),
        "log2fc": result.get("log2foldchanges") or [np.nan] * len(features),
        "padj": result.get("pvals_adj") or [np.nan] * len(features),
        "pct_A": result.get("pct_source") or [np.nan] * len(features),
        "pct_B": result.get("pct_reference") or [np.nan] * len(features),
    })


def plot_simple_design(result, title, padj_cutoff, log2fc_cutoff):
    df = result_dataframe(result)
    if df.empty:
        print(f"No rows to plot for {title}")
        return
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["log2fc", "padj"])
    sig = (df["padj"].astype(float) < float(padj_cutoff)) & (df["log2fc"].astype(float).abs() >= float(log2fc_cutoff))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(np.log10(pd.to_numeric(df["base_mean"], errors="coerce").fillna(0) + 1), df["log2fc"], c=np.where(sig, "tab:red", "0.65"), s=10)
    axes[0].axhline(log2fc_cutoff, color="0.2", lw=1, ls="--")
    axes[0].axhline(-log2fc_cutoff, color="0.2", lw=1, ls="--")
    axes[0].set_xlabel("log10(base mean + 1)")
    axes[0].set_ylabel("log2FC")
    axes[0].set_title(f"{title}: MA")
    y = -np.log10(np.clip(pd.to_numeric(df["padj"], errors="coerce"), np.nextafter(0, 1), 1))
    axes[1].scatter(df["log2fc"], y, c=np.where(sig, "tab:red", "0.65"), s=10)
    axes[1].axvline(log2fc_cutoff, color="0.2", lw=1, ls="--")
    axes[1].axvline(-log2fc_cutoff, color="0.2", lw=1, ls="--")
    axes[1].axhline(-math.log10(padj_cutoff), color="0.2", lw=1, ls="--")
    axes[1].set_xlabel("log2FC")
    axes[1].set_ylabel("-log10 adjusted p")
    axes[1].set_title(f"{title}: volcano")
    plt.tight_layout()


plot_simple_design(wilcoxon_result, "Wilcoxon", wilcoxon_padj_cutoff, wilcoxon_log2fc_cutoff)
if deseq2_result is not None:
    plot_simple_design(deseq2_result, "DESeq2", pseudobulk_padj_cutoff, pseudobulk_log2fc_cutoff)


## Save Tables

In [ ]:
wilcoxon_csv = repo_root / "notebooks" / "simple_design_wilcoxon_recreated.csv"
deseq2_csv = repo_root / "notebooks" / "simple_design_deseq2_recreated.csv"
wilcoxon_simple_design_table.to_csv(wilcoxon_csv, index=False)
deseq2_simple_design_table.to_csv(deseq2_csv, index=False)
print(f"Wrote {wilcoxon_csv}")
print(f"Wrote {deseq2_csv}")
